In [0]:
# Day 11 — Window Functions Part 2
# lag, lead, running totals, moving averages - Day 11 is about comparing a row with its neighbours.

In [0]:
from pyspark.sql.functions import (
    col, lag, lead, sum, avg, max, min,
    round, first, last
)
from pyspark.sql.window import Window

# Monthly sales data per region
data = [
    ("North", "2024-01", 50000),
    ("North", "2024-02", 62000),
    ("North", "2024-03", 58000),
    ("North", "2024-04", 71000),
    ("North", "2024-05", 65000),
    ("North", "2024-06", 80000),
    ("South", "2024-01", 45000),
    ("South", "2024-02", 48000),
    ("South", "2024-03", 52000),
    ("South", "2024-04", 49000),
    ("South", "2024-05", 61000),
    ("South", "2024-06", 58000),
]

cols = ["region", "month", "sales"]
df = spark.createDataFrame(data, cols)
display(df)

region,month,sales
North,2024-01,50000
North,2024-02,62000
North,2024-03,58000
North,2024-04,71000
North,2024-05,65000
North,2024-06,80000
South,2024-01,45000
South,2024-02,48000
South,2024-03,52000
South,2024-04,49000


In [0]:
# Look at previous month's sales
windowSpec = Window \
    .partitionBy("region") \
    .orderBy("month")

# lag(column, offset, default)
# offset=1 means 1 row back
# default=0 means use 0 if no previous row exists
df = df.withColumn("prev_month_sales",
    lag("sales", 1, 0).over(windowSpec))

display(df.orderBy("region", "month"))
# First month of each region → prev = 0 (our default)

region,month,sales,prev_month_sales
North,2024-01,50000,0
North,2024-02,62000,50000
North,2024-03,58000,62000
North,2024-04,71000,58000
North,2024-05,65000,71000
North,2024-06,80000,65000
South,2024-01,45000,0
South,2024-02,48000,45000
South,2024-03,52000,48000
South,2024-04,49000,52000


In [0]:
# Look at next month's sales
windowSpec = Window \
    .partitionBy("region") \
    .orderBy("month")

df = df.withColumn("next_month_sales",
    lead("sales", 1, 0).over(windowSpec))

display(df.orderBy("region", "month"))
# Last month of each region → next = 0 (our default)

region,month,sales,prev_month_sales,next_month_sales
North,2024-01,50000,0,62000
North,2024-02,62000,50000,58000
North,2024-03,58000,62000,71000
North,2024-04,71000,58000,65000
North,2024-05,65000,71000,80000
North,2024-06,80000,65000,0
South,2024-01,45000,0,48000
South,2024-02,48000,45000,52000
South,2024-03,52000,48000,49000
South,2024-04,49000,52000,61000


In [0]:
# Real business metric — how much did sales change vs last month?
windowSpec = Window \
    .partitionBy("region") \
    .orderBy("month")

df = df.withColumn("prev_sales",
    lag("sales", 1).over(windowSpec)) \
    .withColumn("mom_change",
        col("sales") - col("prev_sales")) \
    .withColumn("mom_pct_change",
        round((col("sales") - col("prev_sales"))
              / col("prev_sales") * 100, 2))

display(df.select(
    "region","month","sales",
    "prev_sales","mom_change","mom_pct_change"
).orderBy("region","month"))

region,month,sales,prev_sales,mom_change,mom_pct_change
North,2024-01,50000,null,null,null
North,2024-02,62000,50000,12000,24.0
North,2024-03,58000,62000,-4000,-6.45
North,2024-04,71000,58000,13000,22.41
North,2024-05,65000,71000,-6000,-8.45
North,2024-06,80000,65000,15000,23.08
South,2024-01,45000,null,null,null
South,2024-02,48000,45000,3000,6.67
South,2024-03,52000,48000,4000,8.33
South,2024-04,49000,52000,-3000,-5.77


In [0]:
# Running total — sum of all sales up to current row
windowSpec = Window \
    .partitionBy("region") \
    .orderBy("month") \
    .rowsBetween(
        Window.unboundedPreceding,  # from very first row
        Window.currentRow           # to current row
    )

df = df.withColumn("running_total",
    sum("sales").over(windowSpec))

display(df.select(
    "region","month","sales","running_total"
).orderBy("region","month"))

region,month,sales,running_total
North,2024-01,50000,50000
North,2024-02,62000,112000
North,2024-03,58000,170000
North,2024-04,71000,241000
North,2024-05,65000,306000
North,2024-06,80000,386000
South,2024-01,45000,45000
South,2024-02,48000,93000
South,2024-03,52000,145000
South,2024-04,49000,194000


In [0]:
# 3-month moving average — smooth out spikes
windowSpec = Window \
    .partitionBy("region") \
    .orderBy("month") \
    .rowsBetween(-2, 0)  # current + 2 rows before = 3 rows

df = df.withColumn("moving_avg_3m",
    round(avg("sales").over(windowSpec), 0))

display(df.select(
    "region","month","sales","moving_avg_3m"
).orderBy("region","month"))

region,month,sales,moving_avg_3m
North,2024-01,50000,50000.0
North,2024-02,62000,56000.0
North,2024-03,58000,56667.0
North,2024-04,71000,63667.0
North,2024-05,65000,64667.0
North,2024-06,80000,72000.0
South,2024-01,45000,45000.0
South,2024-02,48000,46500.0
South,2024-03,52000,48333.0
South,2024-04,49000,49667.0


In [0]:
# Track highest and lowest sales seen so far
windowSpec = Window \
    .partitionBy("region") \
    .orderBy("month") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

df = df.withColumn("running_max",
    max("sales").over(windowSpec)) \
    .withColumn("running_min",
    min("sales").over(windowSpec))

display(df.select(
    "region","month","sales",
    "running_max","running_min"
).orderBy("region","month"))

region,month,sales,running_max,running_min
North,2024-01,50000,50000,50000
North,2024-02,62000,62000,50000
North,2024-03,58000,62000,50000
North,2024-04,71000,71000,50000
North,2024-05,65000,71000,50000
North,2024-06,80000,80000,50000
South,2024-01,45000,45000,45000
South,2024-02,48000,48000,45000
South,2024-03,52000,52000,45000
South,2024-04,49000,52000,45000


In [0]:
# Get first and last sale in each region's history
windowFull = Window \
    .partitionBy("region") \
    .orderBy("month") \
    .rowsBetween(
        Window.unboundedPreceding,
        Window.unboundedFollowing
    )

df = df.withColumn("first_month_sale",
    first("sales").over(windowFull)) \
    .withColumn("last_month_sale",
    last("sales").over(windowFull))

display(df.select(
    "region","month","sales",
    "first_month_sale","last_month_sale"
).orderBy("region","month"))

region,month,sales,first_month_sale,last_month_sale
North,2024-01,50000,50000,80000
North,2024-02,62000,50000,80000
North,2024-03,58000,50000,80000
North,2024-04,71000,50000,80000
North,2024-05,65000,50000,80000
North,2024-06,80000,50000,80000
South,2024-01,45000,45000,58000
South,2024-02,48000,45000,58000
South,2024-03,52000,45000,58000
South,2024-04,49000,45000,58000


In [0]:
# Real world — complete sales analytics in one query
windowSpec = Window \
    .partitionBy("region") \
    .orderBy("month")

windowRunning = Window \
    .partitionBy("region") \
    .orderBy("month") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

windowMoving = Window \
    .partitionBy("region") \
    .orderBy("month") \
    .rowsBetween(-2, 0)

report = df.withColumn("prev_sales",
    lag("sales", 1).over(windowSpec)) \
    .withColumn("mom_change",
    col("sales") - col("prev_sales")) \
    .withColumn("running_total",
    sum("sales").over(windowRunning)) \
    .withColumn("moving_avg_3m",
    round(avg("sales").over(windowMoving), 0)) \
    .withColumn("running_max",
    max("sales").over(windowRunning))

display(report.orderBy("region", "month"))

# Save as Delta
spark.sql("DROP TABLE IF EXISTS sales_analytics")
report.write.format("delta").mode("overwrite") \
    .saveAsTable("sales_analytics")
print("✅ Sales analytics report saved!")

region,month,sales,prev_month_sales,next_month_sales,prev_sales,mom_change,mom_pct_change,running_total,moving_avg_3m,running_max,running_min,first_month_sale,last_month_sale
North,2024-01,50000,0,62000,null,null,null,50000,50000.0,50000,50000,50000,80000
North,2024-02,62000,50000,58000,50000,12000,24.0,112000,56000.0,62000,50000,50000,80000
North,2024-03,58000,62000,71000,62000,-4000,-6.45,170000,56667.0,62000,50000,50000,80000
North,2024-04,71000,58000,65000,58000,13000,22.41,241000,63667.0,71000,50000,50000,80000
North,2024-05,65000,71000,80000,71000,-6000,-8.45,306000,64667.0,71000,50000,50000,80000
North,2024-06,80000,65000,0,65000,15000,23.08,386000,72000.0,80000,50000,50000,80000
South,2024-01,45000,0,48000,null,null,null,45000,45000.0,45000,45000,45000,58000
South,2024-02,48000,45000,52000,45000,3000,6.67,93000,46500.0,48000,45000,45000,58000
South,2024-03,52000,48000,49000,48000,4000,8.33,145000,48333.0,52000,45000,45000,58000
South,2024-04,49000,52000,61000,52000,-3000,-5.77,194000,49667.0,52000,45000,45000,58000


✅ Sales analytics report saved!
